
Loads clean raw, extracts outcome events, attaches RW metadata from trajectories.csv, saves epochs.

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne

PARTICIPANT_LABEL = 'A'
SUBJECT           = '01'
SESSION           = '01'
TASK              = 'Cannonball MF'

PROJECT_ROOT    = Path('..').resolve()
sub_ses_dir     = PROJECT_ROOT / 'data' / 'derivatives' / 'mne_preproc' / f'sub-{SUBJECT}' / f'ses-{SESSION}'
prefix          = f'sub-{SUBJECT}_ses-{SESSION}_task-{TASK}'

CLEAN_RAW_PATH    = sub_ses_dir / f'Dataset_{PARTICIPANT_LABEL}_clean_raw.fif'
TRAJECTORIES_PATH = PROJECT_ROOT / 'src' / 'trajectories.csv'
EPOCHS_OUT_PATH   = sub_ses_dir / f'{prefix}_epochs_feedback_clean-epo.fif'

EPOCH_TMIN = -0.7
EPOCH_TMAX =  0.8
BASELINE   = (-0.2, 0.0)

raw = mne.io.read_raw_fif(str(CLEAN_RAW_PATH), preload=True, verbose=False)
print(f'Loaded: {CLEAN_RAW_PATH.name}')
print(f'Channels: {len(raw.ch_names)}  |  sfreq: {raw.info["sfreq"]} Hz')
print(f'Duration: {raw.times[-1]:.1f} s')

Loaded: Dataset_A_clean_raw.fif
Channels: 67  |  sfreq: 256.0 Hz
Duration: 1740.0 s


## Extract Outcome Events

In [15]:
EVENT_ID = {'loss': 17, 'reward': 19}

if 'Status' in raw.ch_names:
    events_all = mne.find_events(raw, stim_channel='Status', shortest_event=1, verbose=False)
else:
    events_all = mne.find_events(raw, shortest_event=1, verbose=False)

print(f'Total events found: {len(events_all)}')

# Keep only outcome codes 17/19
known_codes = set(EVENT_ID.values())
mask = np.isin(events_all[:, 2], list(known_codes))
outcome_events = events_all[mask]
code_to_name   = {v: k for k, v in EVENT_ID.items()}

print(f'Outcome events: {len(outcome_events)}')
for code in sorted(np.unique(outcome_events[:, 2])):
    n = np.sum(outcome_events[:, 2] == code)
    print(f'  {code} ({code_to_name[code]}): {n}')

Total events found: 3599
Outcome events: 984
  17 (loss): 525
  19 (reward): 459


## Load RW Metadata from Trajectories

In [16]:
traj   = pd.read_csv(TRAJECTORIES_PATH)
traj_p = traj[traj['participant_id'] == PARTICIPANT_LABEL].reset_index(drop=True)

assert len(traj_p) > 0, f'No data for participant {PARTICIPANT_LABEL}'
print(f'Trajectory rows for {PARTICIPANT_LABEL}: {len(traj_p)}')

n_events = len(outcome_events)
n_traj   = len(traj_p)
if n_events != n_traj:
    print(f'WARNING: {n_events} events vs {n_traj} trajectory rows — truncating to shorter')
    n = min(n_events, n_traj)
    outcome_events = outcome_events[:n]
    traj_p = traj_p.iloc[:n].reset_index(drop=True)
else:
    print(f'Alignment OK: {n_events} events == {n_traj} rows')

metadata = traj_p.copy()
metadata['eeg_onset']  = outcome_events[:, 0] / raw.info['sfreq']
metadata['eeg_sample'] = outcome_events[:, 0]
metadata['eeg_code']   = outcome_events[:, 2]
metadata['eeg_event']  = [code_to_name[c] for c in outcome_events[:, 2]]
print(f'Metadata columns: {list(metadata.columns)}')

Trajectory rows for A: 720
Metadata columns: ['participant_id', 'trial', 'value_0', 'value_1', 'expected_value', 'prediction_error', 'value_uncertainty', 'eeg_onset', 'eeg_sample', 'eeg_code', 'eeg_event']


## Create and Save Epochs

In [17]:
epochs = mne.Epochs(
    raw,
    outcome_events,
    event_id  = EVENT_ID,
    tmin      = EPOCH_TMIN,
    tmax      = EPOCH_TMAX,
    baseline  = BASELINE,
    metadata  = metadata,
    preload   = True,
    reject    = None,
    reject_by_annotation = True,
    verbose   = False,
)

print(f'Epochs created : {len(epochs)}')
print(f'Conditions     : {dict({k: len(epochs[k]) for k in epochs.event_id})}')
print(f'Metadata       : {epochs.metadata is not None}')

epochs.save(str(EPOCHS_OUT_PATH), overwrite=True)
print(f'Saved: {EPOCHS_OUT_PATH}')

Epochs created : 720
Conditions     : {'loss': 375, 'reward': 345}
Metadata       : True
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Saved: /Users/arthurhsia/Desktop/Psychology/EEG/data/derivatives/mne_preproc/sub-01/ses-01/sub-01_ses-01_task-Cannonball MF_epochs_feedback_clean-epo.fif
